# Phase 2 Baseline Vision Model

Goal: use a pretrained chest X-ray model before fine-tuning.
This gives us a baseline to compare against later.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
torch.__version__, torch.cuda.is_available()

('2.13.0+cpu', False)

In [3]:
PROJECT_ROOT = Path("..").resolve()
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "manifest.parquet"

df = pd.read_parquet(MANIFEST_PATH)
test_df = df[df["split"] == "test"].copy()

df.shape, test_df.shape, test_df["study_id"].nunique()

((3791, 11), (564, 11), 550)

In [4]:
DISEASE_LABELS = [
    "Cardiomegaly",
    "Atelectasis",
    "Consolidation / Pneumonia",
    "Pleural Effusion",
    "Edema",
    "Pneumothorax",
]

DISEASE_LABELS

['Cardiomegaly',
 'Atelectasis',
 'Consolidation / Pneumonia',
 'Pleural Effusion',
 'Edema',
 'Pneumothorax']

In [5]:
def make_target(labels):
    return [1 if label in labels else 0 for label in DISEASE_LABELS]

test_df["target"] = test_df["labels"].apply(make_target)
test_df[["labels", "target"]].head()

,labels,target
0,[No Finding],"[0, 0, 0, 0, 0, 0]"
1,[No Finding],"[0, 0, 0, 0, 0, 0]"
2,[No Finding],"[0, 0, 0, 0, 0, 0]"
3,[Other],"[0, 0, 0, 0, 0, 0]"
4,[Other],"[0, 0, 0, 0, 0, 0]"


In [6]:
import torchxrayvision as xrv

In [8]:
class CXRDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image_path = PROJECT_ROOT / Path(str(row["image_path"]).replace("\\", "/"))

        img = Image.open(image_path).convert("L")
        img = np.array(img).astype(np.float32)

        # torchxrayvision expects values roughly in [-1024, 1024]
        img = xrv.datasets.normalize(img, 255)
        img = img[None, :, :]  # 1 channel

        target = np.array(row["target"], dtype=np.float32)

        return {
            "image": torch.from_numpy(img),
            "target": torch.from_numpy(target),
            "study_id": row["study_id"],
            "image_id": row["image_id"],
        }

In [9]:
test_ds = CXRDataset(test_df)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

batch = next(iter(test_loader))
batch["image"].shape, batch["target"].shape

(torch.Size([16, 1, 224, 224]), torch.Size([16, 6]))

In [10]:
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.pathologies

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O C:\Users\rashe\.torchxrayvision\models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]


['Atelectasis',
 'Consolidation',
 'Infiltration',
 'Pneumothorax',
 'Edema',
 'Emphysema',
 'Fibrosis',
 'Effusion',
 'Pneumonia',
 'Pleural_Thickening',
 'Cardiomegaly',
 'Nodule',
 'Mass',
 'Hernia',
 'Lung Lesion',
 'Fracture',
 'Lung Opacity',
 'Enlarged Cardiomediastinum']

In [11]:
list(model.pathologies)

['Atelectasis',
 'Consolidation',
 'Infiltration',
 'Pneumothorax',
 'Edema',
 'Emphysema',
 'Fibrosis',
 'Effusion',
 'Pneumonia',
 'Pleural_Thickening',
 'Cardiomegaly',
 'Nodule',
 'Mass',
 'Hernia',
 'Lung Lesion',
 'Fracture',
 'Lung Opacity',
 'Enlarged Cardiomediastinum']

In [12]:
XRV_LABEL_MAP = {
    "Cardiomegaly": "Cardiomegaly",
    "Atelectasis": "Atelectasis",
    "Consolidation / Pneumonia": "Consolidation",
    "Pleural Effusion": "Effusion",
    "Edema": "Edema",
    "Pneumothorax": "Pneumothorax",
}

xrv_indices = {
    our_label: list(model.pathologies).index(xrv_label)
    for our_label, xrv_label in XRV_LABEL_MAP.items()
}

xrv_indices

{'Cardiomegaly': 10,
 'Atelectasis': 0,
 'Consolidation / Pneumonia': 1,
 'Pleural Effusion': 7,
 'Edema': 4,
 'Pneumothorax': 3}

In [13]:
rows = []

with torch.no_grad():
    for batch in test_loader:
        images = batch["image"].to(device)
        outputs = model(images)

        probs = outputs[:, list(xrv_indices.values())].cpu().numpy()
        targets = batch["target"].numpy()

        for i in range(len(probs)):
            row = {
                "study_id": batch["study_id"][i],
                "image_id": batch["image_id"][i],
            }

            for j, label in enumerate(DISEASE_LABELS):
                safe = label.replace(" / ", "_").replace(" ", "_")
                row[f"true_{safe}"] = targets[i, j]
                row[f"prob_{safe}"] = probs[i, j]

            rows.append(row)

pred_df = pd.DataFrame(rows)
pred_df.head()

,study_id,image_id,true_Cardiomegaly,prob_Cardiomegaly,true_Atelectasis,prob_Atelectasis,true_Consolidation_Pneumonia,prob_Consolidation_Pneumonia,true_Pleural_Effusion,prob_Pleural_Effusion,true_Edema,prob_Edema,true_Pneumothorax,prob_Pneumothorax
0,1,1_IM-0001-4001.dcm,0.0,0.018473,0.0,0.005424,0.0,0.092092,0.0,0.006444,0.0,0.000992,0.0,0.026190
1,1009,1009_IM-0010-1001.dcm,0.0,0.052824,0.0,0.048600,0.0,0.035297,0.0,0.024251,0.0,0.001874,0.0,0.137431
2,1010,1010_IM-0012-1001.dcm,0.0,0.026977,0.0,0.043934,0.0,0.076394,0.0,0.027565,0.0,0.003855,0.0,0.110184
3,1018,1018_IM-0014-5001.dcm,0.0,0.061428,0.0,0.249015,0.0,0.108861,0.0,0.036233,0.0,0.016387,0.0,0.058699
4,1020,1020_IM-0017-1001.dcm,0.0,0.091307,0.0,0.176351,0.0,0.110026,0.0,0.031733,0.0,0.056399,0.0,0.030630


In [14]:
prob_cols = [c for c in pred_df.columns if c.startswith("prob_")]
true_cols = [c for c in pred_df.columns if c.startswith("true_")]

study_preds = (
    pred_df
    .groupby("study_id")[prob_cols]
    .mean()
    .reset_index()
)

study_true = (
    pred_df
    .groupby("study_id")[true_cols]
    .max()
    .reset_index()
)

study_df = study_true.merge(study_preds, on="study_id")
study_df.shape

(550, 13)

In [15]:
metric_rows = []

for label in DISEASE_LABELS:
    safe = label.replace(" / ", "_").replace(" ", "_")
    y_true = study_df[f"true_{safe}"]
    y_prob = study_df[f"prob_{safe}"]

    if y_true.nunique() < 2:
        auroc = None
    else:
        auroc = roc_auc_score(y_true, y_prob)

    ap = average_precision_score(y_true, y_prob)

    metric_rows.append({
        "label": label,
        "test_positive_studies": int(y_true.sum()),
        "AUROC": auroc,
        "AP": ap,
        "note": "data-limited" if label in ["Edema", "Pneumothorax"] else "",
    })

metrics = pd.DataFrame(metric_rows)
metrics

,label,test_positive_studies,AUROC,AP,note
0,Cardiomegaly,44,0.896065,0.423952,
1,Atelectasis,44,0.816475,0.305423,
2,Consolidation / Pneumonia,28,0.895183,0.293812,
3,Pleural Effusion,22,0.931129,0.705717,
4,Edema,2,0.552920,0.168707,data-limited
5,Pneumothorax,3,0.329677,0.005801,data-limited


In [16]:
OUT_DIR = PROJECT_ROOT / "outputs" / "vision"
OUT_DIR.mkdir(parents=True, exist_ok=True)

study_df.to_csv(OUT_DIR / "baseline_test_predictions.csv", index=False)
metrics.to_csv(OUT_DIR / "baseline_test_metrics.csv", index=False)

OUT_DIR

WindowsPath('E:/Project/RadScribe/outputs/vision')

## Baseline Result Notes

This notebook used a pretrained torchxrayvision DenseNet model.

No training was done.

Predictions were averaged at study level before evaluation.

Edema and Pneumothorax are data-limited because the test split has very few positive studies.